# State Space Models (SSMs) — A Complete Working Explanation

SSMs are the architecture behind **Mamba**, **S4**, and **H3** — sequence models that rival Transformers on long sequences while being much more efficient.

This notebook walks through:
1. What an SSM is mathematically
2. The continuous-time formulation
3. The discrete-time recurrence (how it runs)
4. The convolutional view (how it trains efficiently)
5. Selective SSMs (Mamba's key innovation)
6. Side-by-side comparison with Transformers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import expm

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Libraries loaded.')

---
## 1. The Core Idea

An SSM maps an input sequence $u(t)$ to an output sequence $y(t)$ **through a hidden state** $x(t)$.

Think of the hidden state as a **compressed memory** of everything seen so far — like a fixed-size RAM buffer that gets updated at every step.

### Continuous-time equations

$$\dot{x}(t) = \mathbf{A}\, x(t) + \mathbf{B}\, u(t)$$
$$y(t) = \mathbf{C}\, x(t) + \mathbf{D}\, u(t)$$

| Symbol | Shape | Role |
|--------|-------|------|
| $x(t)$ | $(N,)$ | Hidden state — the model's memory |
| $u(t)$ | $(1,)$ | Input at time $t$ |
| $y(t)$ | $(1,)$ | Output at time $t$ |
| $\mathbf{A}$ | $(N \times N)$ | State transition — how memory evolves |
| $\mathbf{B}$ | $(N \times 1)$ | Input projection — how input enters state |
| $\mathbf{C}$ | $(1 \times N)$ | Output projection — how state becomes output |
| $\mathbf{D}$ | scalar | Skip connection (often set to 0) |

> **Key insight**: $\mathbf{A}$, $\mathbf{B}$, $\mathbf{C}$ are *learned parameters* — the model discovers how to store and retrieve relevant information from its state.

---
## 2. Discretisation — From Continuous to Computable

Neural networks process discrete tokens, not continuous functions. We discretise using the **Zero-Order Hold (ZOH)** method with a timescale parameter $\Delta$ (also learned):

$$\bar{\mathbf{A}} = e^{\Delta \mathbf{A}}$$
$$\bar{\mathbf{B}} = (\Delta \mathbf{A})^{-1}(e^{\Delta \mathbf{A}} - I) \cdot \Delta \mathbf{B}$$

Now the SSM becomes a **recurrence** we can compute step by step:

$$x_k = \bar{\mathbf{A}}\, x_{k-1} + \bar{\mathbf{B}}\, u_k$$
$$y_k = \mathbf{C}\, x_k$$

This is efficient for **inference** — $O(L)$ time and $O(N)$ memory regardless of sequence length.

In [ ]:
def discretise(A, B, delta):
    """ZOH discretisation of continuous SSM matrices."""
    N = A.shape[0]
    dA = delta * A
    A_bar = expm(dA)                          # matrix exponential
    B_bar = np.linalg.solve(dA, (A_bar - np.eye(N))) @ (delta * B)
    return A_bar, B_bar


def ssm_recurrence(A_bar, B_bar, C, u_seq):
    """Run SSM as a recurrence. Returns hidden states and outputs."""
    N = A_bar.shape[0]
    L = len(u_seq)
    x = np.zeros(N)
    ys, xs = [], []
    for u in u_seq:
        x = A_bar @ x + B_bar * u
        y = C @ x
        xs.append(x.copy())
        ys.append(float(y))
    return np.array(xs), np.array(ys)


# --- Build a tiny SSM (N=4 state dims) ---
np.random.seed(42)
N = 4

# HiPPO-inspired A: structured matrix that preserves history
A = -np.diag(np.arange(1, N+1), 0).astype(float)
B = np.random.randn(N, 1)
C = np.random.randn(1, N)
delta = 0.1

A_bar, B_bar = discretise(A, B.flatten(), delta)

# Input: a noisy sine wave
L = 200
t = np.linspace(0, 4 * np.pi, L)
u = np.sin(t) + 0.3 * np.random.randn(L)

xs, ys = ssm_recurrence(A_bar, B_bar, C, u)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(t, u, color='#378ADD', alpha=0.7, lw=1)
axes[0].set_title('Input signal u(t)', fontweight='bold')
axes[0].set_xlabel('time')

for i in range(N):
    axes[1].plot(t, xs[:, i], label=f'$x_{i+1}$', lw=1)
axes[1].set_title('Hidden state x(t) — N=4 dims', fontweight='bold')
axes[1].set_xlabel('time')
axes[1].legend(fontsize=8)

axes[2].plot(t, ys, color='#1D9E75', lw=1.5)
axes[2].set_title('Output y(t)', fontweight='bold')
axes[2].set_xlabel('time')

plt.tight_layout()
plt.suptitle('SSM recurrence mode', y=1.02, fontsize=13, fontweight='bold')
plt.show()
print(f'State matrix A_bar shape: {A_bar.shape}')
print(f'Computed {L} steps. Memory per step: {N} floats (constant!)')

---
## 3. The Convolutional View — Fast Training

Unrolling the recurrence reveals something beautiful. Substituting $x_k = \bar{A} x_{k-1} + \bar{B} u_k$ repeatedly:

$$y_k = \sum_{j=0}^{k} \mathbf{C}\, \bar{\mathbf{A}}^{k-j}\, \bar{\mathbf{B}}\, u_j$$

This is a **convolution** $y = \bar{K} * u$ where the kernel is:

$$\bar{K} = (\mathbf{C}\bar{\mathbf{B}},\; \mathbf{C}\bar{\mathbf{A}}\bar{\mathbf{B}},\; \mathbf{C}\bar{\mathbf{A}}^2\bar{\mathbf{B}},\; \ldots)$$

Using **Fast Fourier Transform**, this convolution runs in $O(L \log L)$ — parallelisable across the entire sequence. That's why SSMs can train fast like Transformers but run fast like RNNs.

| Mode | Time | Memory | Parallelisable |
|------|------|--------|----------------|
| Recurrence | $O(L)$ | $O(N)$ | No (sequential) |
| Convolution | $O(L \log L)$ | $O(L)$ | **Yes** |
| Transformer attention | $O(L^2)$ | $O(L^2)$ | Yes |

In [ ]:
def ssm_kernel(A_bar, B_bar, C, L):
    """Compute the SSM convolution kernel K of length L."""
    K = []
    power = np.eye(A_bar.shape[0])
    for _ in range(L):
        K.append(float(C @ power @ B_bar))
        power = power @ A_bar
    return np.array(K)


def ssm_conv(K, u):
    """Apply SSM as a causal convolution via FFT."""
    L = len(u)
    # Causal conv: zero-pad and use FFT
    fft_len = 2 * L
    K_fft = np.fft.rfft(K, n=fft_len)
    u_fft = np.fft.rfft(u, n=fft_len)
    y_full = np.fft.irfft(K_fft * u_fft)
    return y_full[:L]


K = ssm_kernel(A_bar, B_bar.reshape(-1), C.reshape(-1), L)
ys_conv = ssm_conv(K, u)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

axes[0].stem(np.arange(40), K[:40], linefmt='#534AB7', markerfmt='o',
             basefmt='gray', use_line_collection=True)
axes[0].set_title('SSM convolution kernel K (first 40 steps)', fontweight='bold')
axes[0].set_xlabel('lag')
axes[0].set_ylabel('weight')

axes[1].plot(t, ys, color='#378ADD', lw=2, label='Recurrence output', alpha=0.7)
axes[1].plot(t, ys_conv, color='#D85A30', lw=1.5, linestyle='--',
             label='Conv output', alpha=0.9)
axes[1].set_title('Recurrence vs convolution (should be identical)', fontweight='bold')
axes[1].set_xlabel('time')
axes[1].legend()

plt.tight_layout()
plt.show()

max_diff = np.max(np.abs(ys - ys_conv))
print(f'Max difference between recurrence and conv outputs: {max_diff:.2e} (should be ~0)')

---
## 4. HiPPO — The Smart State Matrix

A random $\mathbf{A}$ doesn't work well — the state either explodes or vanishes. **HiPPO** (High-order Polynomial Projection Operators) provides a principled $\mathbf{A}$ that provably compresses history optimally.

The HiPPO-LegS matrix projects the input history onto Legendre polynomials, giving each state dimension a role in remembering a different time-scale:

$$A_{nk} = -\begin{cases}(2n+1)^{1/2}(2k+1)^{1/2} & n > k \\ n+1 & n = k \\ 0 & n < k\end{cases}$$

The result: the hidden state at time $t$ encodes the *entire past* as Legendre polynomial coefficients.

In [ ]:
def make_hippo(N):
    """Construct the HiPPO-LegS A and B matrices."""
    A = np.zeros((N, N))
    for n in range(N):
        for k in range(N):
            if n > k:
                A[n, k] = -np.sqrt((2*n+1) * (2*k+1))
            elif n == k:
                A[n, k] = -(n+1)
    B = np.array([np.sqrt(2*n+1) for n in range(N)])
    return A, B


N = 64
A_hippo, B_hippo = make_hippo(N)
C_hippo = np.random.randn(1, N) * 0.1
delta_hippo = 0.01

A_bar_h, B_bar_h = discretise(A_hippo, B_hippo, delta_hippo)

# Compare random A vs HiPPO A on memory task:
# input = impulse at t=0, does the model remember it?
L_mem = 500
impulse = np.zeros(L_mem)
impulse[0] = 1.0

# Random stable A
A_rand = -np.abs(np.random.randn(N, N)) * 0.05
B_rand = np.random.randn(N)
A_bar_r, B_bar_r = discretise(A_rand, B_rand, delta_hippo)

_, ys_hippo = ssm_recurrence(A_bar_h, B_bar_h, C_hippo, impulse)
_, ys_rand = ssm_recurrence(A_bar_r, B_bar_r, C_hippo, impulse)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

axes[0].imshow(np.abs(A_hippo), aspect='auto', cmap='Blues')
axes[0].set_title('HiPPO A matrix (absolute values)', fontweight='bold')
axes[0].set_xlabel('column k')
axes[0].set_ylabel('row n')
plt.colorbar(axes[0].images[0], ax=axes[0])

axes[1].plot(np.abs(ys_rand), color='#E24B4A', lw=1.5,
             label='Random A — forgets quickly', alpha=0.8)
axes[1].plot(np.abs(ys_hippo), color='#1D9E75', lw=1.5,
             label='HiPPO A — retains impulse', alpha=0.9)
axes[1].set_title('Impulse response: does the model remember?', fontweight='bold')
axes[1].set_xlabel('steps after impulse')
axes[1].set_ylabel('|output|')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 5. Selective SSMs — Mamba's Key Innovation

Classical SSMs (S4) have **time-invariant** parameters: $\mathbf{A}$, $\mathbf{B}$, $\mathbf{C}$ are fixed across all time steps. This makes them a static filter — they can't choose *what* to remember based on *content*.

**Mamba** makes $\mathbf{B}$, $\mathbf{C}$, and $\Delta$ **input-dependent**:

$$\Delta_k,\; \mathbf{B}_k,\; \mathbf{C}_k = f(u_k)$$

This means at each step the model can:
- **Open the gate** ($\Delta$ large) → absorb input fully into state
- **Close the gate** ($\Delta$ small) → ignore input, preserve state
- **Focus on relevant tokens** ($\mathbf{B}_k$, $\mathbf{C}_k$) → context-dependent selection

This breaks the convolution view (since $K$ is no longer fixed), but enables **content-aware filtering** — the SSM equivalent of attention.

In [ ]:
def selective_ssm_demo(u_seq, A_base, B_base, C_base, delta_fn):
    """
    Simplified selective SSM: delta is input-dependent.
    delta_fn(u) returns a scalar timescale for this token.
    """
    N = A_base.shape[0]
    x = np.zeros(N)
    ys, deltas = [], []
    for u in u_seq:
        delta = delta_fn(u)
        A_bar, B_bar = discretise(A_base, B_base, delta)
        x = A_bar @ x + B_bar * u
        y = C_base @ x
        ys.append(float(y))
        deltas.append(delta)
    return np.array(ys), np.array(deltas)


N_s = 8
A_s, B_s = make_hippo(N_s)
C_s = np.random.randn(1, N_s) * 0.3

# Input: noisy signal with a brief "important" burst in the middle
L_s = 300
t_s = np.linspace(0, 1, L_s)
u_s = 0.2 * np.random.randn(L_s)
u_s[120:140] = 3.0   # important burst

# Static SSM: fixed delta
delta_static = 0.05
A_bar_s, B_bar_s = discretise(A_s, B_s, delta_static)
_, ys_static = ssm_recurrence(A_bar_s, B_bar_s, C_s, u_s)

# Selective SSM: delta scales with |input| — attend to large inputs
def delta_fn(u):
    return 0.01 + 0.2 * np.tanh(abs(u))

ys_sel, deltas = selective_ssm_demo(u_s, A_s, B_s.flatten(), C_s.flatten(), delta_fn)

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

axes[0].plot(t_s, u_s, color='#378ADD', lw=1)
axes[0].axvspan(t_s[120], t_s[140], color='#FAC775', alpha=0.4, label='Important burst')
axes[0].set_title('Input signal', fontweight='bold')
axes[0].legend(fontsize=8)

axes[1].plot(t_s, deltas, color='#D85A30', lw=1.2)
axes[1].axvspan(t_s[120], t_s[140], color='#FAC775', alpha=0.4)
axes[1].set_title('Delta Δ(t) — input-dependent gate (selective SSM)', fontweight='bold')
axes[1].set_ylabel('Δ value')

axes[2].plot(t_s, ys_static, color='#888780', lw=1.2, label='Static SSM', alpha=0.7)
axes[2].plot(t_s, ys_sel, color='#1D9E75', lw=1.5, label='Selective SSM (Mamba-style)')
axes[2].axvspan(t_s[120], t_s[140], color='#FAC775', alpha=0.4)
axes[2].set_title('Output comparison — selective SSM reacts sharply', fontweight='bold')
axes[2].set_xlabel('time')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 6. Complexity Comparison — SSM vs Transformer

This is the main reason SSMs matter for long sequences.

In [ ]:
seq_lengths = np.logspace(2, 5, 50)  # 100 to 100,000 tokens

# Relative compute (normalised at L=1000)
ref = 1000
transformer = (seq_lengths ** 2) / (ref ** 2)
ssm_conv_cost = (seq_lengths * np.log2(seq_lengths)) / (ref * np.log2(ref))
ssm_recur = seq_lengths / ref

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax in axes:
    ax.plot(seq_lengths, transformer, color='#E24B4A', lw=2.5, label='Transformer O(L²)')
    ax.plot(seq_lengths, ssm_conv_cost, color='#534AB7', lw=2,
            label='SSM conv O(L log L)', linestyle='--')
    ax.plot(seq_lengths, ssm_recur, color='#1D9E75', lw=2,
            label='SSM recurrence O(L)', linestyle=':')
    ax.axvline(1000, color='gray', lw=0.8, linestyle='--', alpha=0.5)
    ax.set_xlabel('Sequence length L')
    ax.set_ylabel('Relative compute')
    ax.legend(fontsize=8)

axes[0].set_title('Linear scale', fontweight='bold')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('Log-log scale', fontweight='bold')

plt.suptitle('Compute scaling: SSM vs Transformer', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('At L=100,000:')
L = 100_000
print(f'  Transformer: {L**2 / ref**2:,.0f}x more compute than at L=1000')
print(f'  SSM conv:    {L*np.log2(L)/(ref*np.log2(ref)):.0f}x more compute than at L=1000')
print(f'  SSM recur:   {L/ref:.0f}x more compute than at L=1000')

---
## 7. The Full SSM Architecture in a Real Model

In practice, an SSM layer (like S4 or Mamba) is used inside a deep network:

```
Input tokens
    │
    ▼
Embedding layer          # token → vector
    │
    ▼
┌─────────────────────┐
│  SSM Block × L      │  repeat L times
│  ┌───────────────┐  │
│  │  LayerNorm    │  │
│  │  SSM layer    │  │  ← A, B, C, Δ here
│  │  Gating (×)   │  │  ← SiLU gated MLP
│  │  Residual (+) │  │
│  └───────────────┘  │
└─────────────────────┘
    │
    ▼
LM head → next token probabilities
```

Mamba adds a **hardware-aware parallel scan** (associative scan) that:
- Runs the selective recurrence in $O(\log L)$ depth on GPU
- Avoids materialising the full state sequence in HBM
- Uses kernel fusion to keep everything in SRAM

---
## 8. Summary — Three Views of the Same Model

| View | Equation | When used | Why |
|------|----------|-----------|-----|
| **Continuous** | $\dot{x} = Ax + Bu$ | Theory / initialisation | Connects to classical control theory |
| **Recurrence** | $x_k = \bar{A}x_{k-1} + \bar{B}u_k$ | Inference / generation | $O(N)$ memory, $O(1)$ per step |
| **Convolution** | $y = \bar{K} * u$ | Training | Parallelisable via FFT, $O(L \log L)$ |

### SSM family tree

```
Classical SSMs (1960s)
    │
    ├── LSSL (2021) — first deep SSM
    │
    ├── S4 (2022) — HiPPO + diagonal + conv training
    │       └── S4D, DSS, GSS variants
    │
    ├── H3 (2023) — hybrid SSM + attention
    │
    └── Mamba (2024) — selective SSM, hardware-aware
            └── Mamba-2, Jamba (SSM + Transformer hybrid)
```

### When SSMs beat Transformers
- Very long sequences (> 10k tokens): audio, genomics, time-series
- Streaming / real-time inference (recurrence mode)
- Memory-constrained deployment

### When Transformers still win
- In-context learning and retrieval (attention is more flexible)
- Tasks requiring precise copying from context
- Short sequences where quadratic cost is negligible

In [ ]:
# Final demo: SSM as a learnable smoothing filter
# Shows how A, B, C shape the impulse response / frequency response

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    ('Low-pass (smooth)', -0.5,  '#378ADD'),
    ('Band-pass (resonant)', -0.1, '#D85A30'),
    ('High-pass (edge detect)', -2.0, '#1D9E75'),
]

L_ir = 100
impulse_ir = np.zeros(L_ir); impulse_ir[0] = 1.0

for ax, (label, a_val, color) in zip(axes, configs):
    N_d = 8
    A_d = np.diag([a_val] * N_d)
    B_d = np.ones(N_d)
    C_d = np.random.randn(1, N_d) * 0.5
    Ab, Bb = discretise(A_d, B_d, 0.1)
    _, ir = ssm_recurrence(Ab, Bb, C_d, impulse_ir)
    ax.plot(ir, color=color, lw=2)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('lag')
    ax.axhline(0, color='gray', lw=0.5)

plt.suptitle('SSM impulse responses — different A structures give different memory shapes',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print('\nNotebook complete. Key takeaway:')
print('SSMs are learnable linear recurrences with a dual convolution form.')
print('Selective SSMs (Mamba) add input-dependent parameters for content-aware filtering.')